In [1]:
# preprocess.py

import pandas as pd
import numpy as np
import pickle
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

# ── 1. Load the CSV ──────────────────────────────────────────
df = pd.read_csv("customer_data.csv")
print("Loaded data shape:", df.shape)
print(df.head(3))  # preview first 3 rows so you can see what you're working with

# ── 2. Define features ───────────────────────────────────────
FEATURES = ["pages_viewed", "time_spent_min", "items_in_cart",
            "discount_clicked", "returned_visit"]
SEQ_LEN = 7

# ── 3. Scale features to 0–1 ─────────────────────────────────
scaler = MinMaxScaler()
df[FEATURES] = scaler.fit_transform(df[FEATURES])
print("\nScaling done. Sample scaled values:")
print(df[FEATURES].head(3))

# ── 4. Build sequences ───────────────────────────────────────
X, y = [], []

for cust_id in df["customer_id"].unique():
    cust_df = df[df["customer_id"] == cust_id].reset_index(drop=True)
    for i in range(SEQ_LEN, len(cust_df)):
        X.append(cust_df[FEATURES].iloc[i - SEQ_LEN:i].values)
        y.append(cust_df["will_buy"].iloc[i])

X = np.array(X)
y = np.array(y)
print(f"\nTotal sequences built: {len(X)}")
print(f"X shape: {X.shape}  →  (sequences, days, features)")
print(f"y shape: {y.shape}  →  (sequences,)")

# ── 5. Train / test split ────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"\nTrain size: {X_train.shape[0]} sequences")
print(f"Test size:  {X_test.shape[0]} sequences")

# ── 6. Save arrays to disk ───────────────────────────────────
np.save("X_train.npy", X_train)
np.save("X_test.npy",  X_test)
np.save("y_train.npy", y_train)
np.save("y_test.npy",  y_test)

# ── 7. Save the scaler ───────────────────────────────────────
with open("scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

print("\n✅ All files saved:")
print("   X_train.npy, X_test.npy, y_train.npy, y_test.npy, scaler.pkl")

Loaded data shape: (30000, 10)
   customer_id  day  pages_viewed  time_spent_min  items_in_cart  \
0            0    0             4            0.85              2   
1            0    1             2            6.40              5   
2            0    2             2            2.83              0   

   discount_clicked  returned_visit  cart_trend  time_trend  will_buy  
0                 0               0    2.000000       0.850         0  
1                 0               0    3.500000       3.625         1  
2                 0               0    2.333333       3.360         0  

Scaling done. Sample scaled values:
   pages_viewed  time_spent_min  items_in_cart  discount_clicked  \
0      0.307692        0.016909            0.4               0.0   
1      0.153846        0.127313            1.0               0.0   
2      0.153846        0.056296            0.0               0.0   

   returned_visit  
0             0.0  
1             0.0  
2             0.0  

Total sequences b